## PyTorch NN Module and Optim Module

Using this PyTorch NN Module and torch.optim module , we will improve the neural network Training pipeline.
Earlier we have wrote many of the layer of the neural network manually , we can minimize the manual effort by using these modules.

### NN Module : torch.nn

The torch.nn module in PyTorch is a core library that provides a wide array of classes and functions designed to help developers build neural networks efficiently and effectively. It abstracts the complexity of creating and training neural networks by offering pre-built layers, loss functions, activation functions, and other utilities, enabling you to focus on designing and experimenting with model architectures.

#### Key Components of torch.nn :
**1. Modules(Layers):**
  - *nn.module* : The base class for all neural network modules. Your custome models and layers should inherit or subclass this class.
  - *Common Layers* : It includes layers like nn.Linear(fully connected layer), nn.Conv2d(Convulutional layer), nn.LSTM(recurrent layer) and many others.

**2. Activation Functions:**
  - Functions like nn.ReLU, nn.sigmoid and nn.Tanh introduce non-linearities to the model, allowing it to learn complex patterns.

**3. Loss Functions:**
  - Provides loss functions such as nn.CrossEntropyLoss, nn.MSELoss and nn.NLLLoss to quantify the difference between the model's predictions and the actual targets/labels.

**4. Container Modules:**
  - *nn.sequential* : A Sequntial container to stack layers in order

**5. Regularization and Dropout:**
  - Layers like nn.Dropout and nn.BatchNorm2d help prevent overfitting and improve the model's ability to generalize to new data

### Optim Module : torch.optim

  - ***torch.optim*** is a module in PyTorch that provides a variety of optimization algorithms used to update the parameters of our model during training
  - It Includes common optimizers like Stochastic Gradient Descent(SGD), Adam, RMSprop, and more
  - It handles weight updates efficiently , including additional features like learning rate scheduling and weight decay(regularization)

The model.parameters() method in PyTorch retrives an iterator , which iterator over all the trainable parameters(like weights and biases) in a model. These parameters are instances of torch.nn.Paramter and include
  - *Weights* : The weight matrices of layers like nn.Linear, nn.Conv2d,.etc.
  - *Biases* : The bias terms of layers(if they exists)

The optimizers uses these parameters to compute gradients and update them during training.

#### 1. Creating a Simple Neural network using torch.nn module
  - It has 5 input features and 1 output

In [18]:
# Imports
import torch
import torch.nn as nn

In [19]:
# Creating a simple neural network model with nn module
# Here we are using a single linear layer and applying the sigmoid activation function on it

class MyModel(nn.Module):
  def __init__(self,num_features):
    super().__init__()

    self.linear = nn.Linear(in_features=num_features,out_features=1)
    self.sigmoid = nn.Sigmoid()

  def forward(self,features):

    # This functions calculates : z = w*x + b
    z = self.linear(features)
    # This function applies activation function i,e sigmoid on z
    y_pred = self.sigmoid(z)

    return y_pred


In [20]:
# Creatig  a simple dummy dataset of 10 rows(entries) and 5 columns (features)
data = torch.rand(10,5)

In [21]:
# Accessing the model
model = MyModel(data.shape[1]) # it pass num_features = 5

In [22]:
# Calling model for forward pass
# model.forward(data) # -> This is also possible
# But this is recommended
model(data)

tensor([[0.5741],
        [0.6071],
        [0.6473],
        [0.6412],
        [0.5611],
        [0.6252],
        [0.6336],
        [0.6457],
        [0.6248],
        [0.6663]], grad_fn=<SigmoidBackward0>)

In [23]:
# Trained Weights
model.linear.weight

Parameter containing:
tensor([[ 0.3782, -0.1577,  0.2605,  0.3798, -0.2281]], requires_grad=True)

In [24]:
# Bias
model.linear.bias

Parameter containing:
tensor([0.1745], requires_grad=True)

In [25]:
!pip install torchinfo

In [26]:
# We can get the summary or info about our model
from torchinfo import summary
summary(model,input_size=(10,5))


Layer (type:depth-idx)                   Output Shape              Param #
MyModel                                  [10, 1]                   --
├─Linear: 1-1                            [10, 1]                   6
├─Sigmoid: 1-2                           [10, 1]                   --
Total params: 6
Trainable params: 6
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

#### 2. Creating a bit complex neural network using torch.nn module

  - Here we will have the 3 layers
    - 1 input layer => It will have 5 input features i,e a linear layer
    - 1 hidden layers(inp=5,out=3) => It will take 5 input features as weights and applies *ReLU* on it and produces 3 output weights
    - 1 output layer(inp=3,out=1) => It will take 3 input from hidden layer and applies  *sigmoid* activation function on it and produces 1 output.

In [27]:
class MyModel2(nn.Module):
  def __init__(self,num_features):
    super().__init__()

    self.linear1 = nn.Linear(in_features=num_features,out_features=3)
    self.relu = nn.ReLU()
    self.linear2 = nn.Linear(in_features=3,out_features=1)
    self.sigmoid = nn.Sigmoid()


  def forward(self,features):
    out = self.linear1(features)
    out = self.relu(out)
    out = self.linear2(out)
    out = self.sigmoid(out)

    return out

In [28]:
# Creating a dataset
data = torch.rand(10,5)

# Defining a model
model2 = MyModel2(data.shape[1])

# Calling the model for forward pass
model2(data)

tensor([[0.6279],
        [0.6137],
        [0.6066],
        [0.6162],
        [0.6189],
        [0.6011],
        [0.6257],
        [0.6245],
        [0.6180],
        [0.5958]], grad_fn=<SigmoidBackward0>)

In [29]:
# Summary or info of the model
from torchinfo import summary

summary(model2,(10,5))

Layer (type:depth-idx)                   Output Shape              Param #
MyModel2                                 [10, 1]                   --
├─Linear: 1-1                            [10, 3]                   18
├─ReLU: 1-2                              [10, 3]                   --
├─Linear: 1-3                            [10, 1]                   4
├─Sigmoid: 1-4                           [10, 1]                   --
Total params: 22
Trainable params: 22
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

###### Creating a model with Sequential Container

In [30]:
from torch.nn.modules.linear import Linear
from torch.nn.modules.activation import ReLU
class MyModel3(nn.Module):

  def __init__(self,num_features):
    super().__init__()

    # Grouping all the layers in a sequential container
    self.network = nn.Sequential(
        nn.Linear(in_features=num_features,out_features=3),
        nn.ReLU(),
        nn.Linear(in_features=3,out_features=1),
        nn.Sigmoid()
    )

  def forward(self,features):
    out = self.network(features)

    return out

In [31]:
# Model with sequential container

data = torch.rand(20,5)
# Defining the model
model3 = MyModel3(data.shape[1])

# Creating the forward pass on the neural network
model3(data)

tensor([[0.4044],
        [0.4046],
        [0.4085],
        [0.4089],
        [0.4100],
        [0.4013],
        [0.4042],
        [0.4079],
        [0.4085],
        [0.4095],
        [0.4086],
        [0.4051],
        [0.4078],
        [0.4072],
        [0.4025],
        [0.4093],
        [0.4096],
        [0.4080],
        [0.4063],
        [0.4090]], grad_fn=<SigmoidBackward0>)

In [32]:
summary(model3,(20,5))

Layer (type:depth-idx)                   Output Shape              Param #
MyModel3                                 [20, 1]                   --
├─Sequential: 1-1                        [20, 1]                   --
│    └─Linear: 2-1                       [20, 3]                   18
│    └─ReLU: 2-2                         [20, 3]                   --
│    └─Linear: 2-3                       [20, 1]                   4
│    └─Sigmoid: 2-4                      [20, 1]                   --
Total params: 22
Trainable params: 22
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

#### Creating a Full Training pipeline of Simple Neural network using nn and optim module on Breast Cancer Dataset

In [41]:
# Imports
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
from torchinfo import summary
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

In [42]:
#Load Datasets
df = pd.read_csv("https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv")
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [43]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 33 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       569 non-null    int64  
 1   diagnosis                569 non-null    object 
 2   radius_mean              569 non-null    float64
 3   texture_mean             569 non-null    float64
 4   perimeter_mean           569 non-null    float64
 5   area_mean                569 non-null    float64
 6   smoothness_mean          569 non-null    float64
 7   compactness_mean         569 non-null    float64
 8   concavity_mean           569 non-null    float64
 9   concave points_mean      569 non-null    float64
 10  symmetry_mean            569 non-null    float64
 11  fractal_dimension_mean   569 non-null    float64
 12  radius_se                569 non-null    float64
 13  texture_se               569 non-null    float64
 14  perimeter_se             5

##### Data Preprocessing

In [44]:
# Removing unnecessary columns
df.drop(columns=['id','Unnamed: 32'],inplace=True)

In [45]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 31 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   diagnosis                569 non-null    object 
 1   radius_mean              569 non-null    float64
 2   texture_mean             569 non-null    float64
 3   perimeter_mean           569 non-null    float64
 4   area_mean                569 non-null    float64
 5   smoothness_mean          569 non-null    float64
 6   compactness_mean         569 non-null    float64
 7   concavity_mean           569 non-null    float64
 8   concave points_mean      569 non-null    float64
 9   symmetry_mean            569 non-null    float64
 10  fractal_dimension_mean   569 non-null    float64
 11  radius_se                569 non-null    float64
 12  texture_se               569 non-null    float64
 13  perimeter_se             569 non-null    float64
 14  area_se                  5

##### train_test split

In [46]:
X_train,X_test,y_train,y_test = train_test_split(df.iloc[:,1:],df.iloc[:,0],test_size=0.2,random_state=1)

print("X_train : ",X_train.shape)
print("X_test : ",X_test.shape)
print("y_train : ",y_train.shape)
print("y_test : ",y_test.shape)

X_train :  (455, 30)
X_test :  (114, 30)
y_train :  (455,)
y_test :  (114,)


##### Scaling

In [47]:
#  Here our data is in different scaling( or range) so to normalize or scale it to a same scale , we will do the scaling on the features input using StandardScaler
print("Before Scaling")
print("X_train",X_train)
print("X_test",X_test)

Before Scaling
X_train      radius_mean  texture_mean  perimeter_mean  area_mean  smoothness_mean  \
408        17.99         20.66          117.80      991.7          0.10360   
4          20.29         14.34          135.10     1297.0          0.10030   
307         9.00         14.40           56.36      246.3          0.07005   
386        12.21         14.09           78.78      462.0          0.08108   
404        12.34         14.95           78.29      469.1          0.08682   
..           ...           ...             ...        ...              ...   
129        19.79         25.12          130.40     1192.0          0.10150   
144        10.75         14.97           68.26      355.3          0.07793   
72         17.20         24.52          114.20      929.4          0.10710   
235        14.03         21.25           89.79      603.4          0.09070   
37         13.03         18.42           82.61      523.8          0.08983   

     compactness_mean  concavity_mean  c

In [48]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.fit_transform(X_test)

In [49]:
print("After Scaling")
print("X_train",X_train)
print("X_test",X_test)

After Scaling
X_train [[ 1.0449852   0.29708512  1.01044815 ...  1.21839781  0.29811854
   0.08380738]
 [ 1.68141074 -1.14098169  1.70338066 ...  0.69716372 -0.88595033
  -0.41234747]
 [-1.44260855 -1.12732916 -1.4504636  ... -1.52233766  0.1807324
  -0.33657109]
 ...
 [ 0.82638686  1.17539807  0.86625411 ...  1.10638475  0.72853438
   3.02284824]
 [-0.05077356  0.43133503 -0.11146165 ... -0.54050588 -1.1207226
  -0.44903285]
 [-0.32748032 -0.21260945 -0.39904866 ... -0.98109057 -1.52732096
  -1.31985978]]
X_test [[ 2.92635643e-01 -1.30427498e+00  4.09740749e-01 ...  1.73162033e-02
  -1.90246784e-01  3.02591603e-01]
 [-1.97911946e-01 -9.58755925e-02 -1.69337819e-01 ...  1.64947589e+00
   1.29351802e+00  1.43498453e+00]
 [-2.68912255e-01 -7.77536786e-01 -3.03699185e-01 ... -6.92881882e-02
   5.74452130e-01  4.67494678e-01]
 ...
 [-1.53433140e+00 -4.39288239e-01 -1.45570598e+00 ...  1.08654734e+00
   1.74708265e+00  1.41744165e+00]
 [ 2.40999055e-01  7.97038058e-02  2.21918697e-01 ... -2

In [50]:
#  Similarly our target labels are categories we will convert it into a numerical label(a binary label as our output labels are only 2 here) because our neural network won't be able to understand this categorical labels

print("Before Label Encoding")
print("y_train")
print(y_train)

Before Label Encoding
y_train
408    M
4      M
307    B
386    B
404    B
      ..
129    M
144    B
72     M
235    B
37     B
Name: diagnosis, Length: 455, dtype: object


In [51]:
print("y_test")
print(y_test)

y_test
421    B
47     M
292    B
186    M
414    M
      ..
172    M
3      M
68     B
448    B
442    B
Name: diagnosis, Length: 114, dtype: object


In [52]:
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.fit_transform(y_test)

In [53]:
print("After Label Encoding")
print("y_train")
print(y_train)
print("y_test")
print(y_test)

After Label Encoding
y_train
[1 1 0 0 0 0 0 1 1 0 0 1 1 0 1 1 0 0 0 1 0 1 1 0 0 0 1 1 0 1 0 1 0 0 0 1 0
 0 0 1 0 0 0 0 0 0 0 0 0 1 0 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0
 1 0 0 0 0 1 1 0 0 1 1 0 1 0 1 0 1 1 0 0 1 1 0 0 0 0 1 0 0 0 0 0 0 0 1 0 1
 0 0 1 1 1 1 0 0 1 0 1 1 1 0 1 0 0 0 0 0 1 0 0 1 1 1 1 0 1 0 0 1 0 1 0 0 0
 1 0 0 0 1 0 0 0 0 1 0 0 1 1 1 0 0 1 0 0 0 0 0 1 0 0 0 1 0 1 1 1 1 0 1 1 1
 0 0 1 0 0 0 0 1 1 0 0 0 1 0 0 0 1 0 0 1 0 0 0 0 1 1 0 1 1 0 1 0 1 1 0 0 1
 0 0 0 0 0 1 0 0 0 0 1 0 0 0 0 0 1 1 1 0 1 1 0 1 0 0 0 1 0 1 0 0 0 0 1 1 0
 0 0 0 1 0 0 1 1 1 0 1 1 1 0 0 0 0 0 0 0 1 0 1 1 1 1 0 0 1 0 0 0 0 0 0 0 1
 1 0 0 0 0 1 0 0 0 0 1 0 0 0 1 0 1 0 0 1 1 1 1 1 0 0 0 0 0 1 0 0 0 0 1 1 0
 0 0 1 0 0 0 1 0 0 1 0 0 1 0 1 0 0 0 0 1 1 0 0 0 1 0 1 1 1 0 0 0 1 0 0 0 1
 1 1 0 1 0 1 1 0 0 0 0 1 1 0 0 0 0 0 1 1 0 1 0 0 0 0 0 1 1 0 0 0 0 1 1 1 0
 0 1 1 1 1 0 1 0 1 1 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 1 1 1 1 0 1 1 0 0 1 1 0
 1 0 0 0 0 0 1 0 1 0 0]
y_test
[0 1 0 1 1 1 1 1 0 0 0 1 1 0 0 0 0 0 0 1

##### Converting this numpy arrays into PyTorch Tensors

In [54]:
# Before Converting its types
print("X_train : ",type(X_train))
print("X_test : ",type(X_test))

print("y_train : ",type(y_train))
print("y_test : ",type(y_test))


X_train :  <class 'numpy.ndarray'>
X_test :  <class 'numpy.ndarray'>
y_train :  <class 'numpy.ndarray'>
y_test :  <class 'numpy.ndarray'>


In [72]:
X_train_tr = torch.from_numpy(X_train).float()
X_test_tr = torch.from_numpy(X_test).float()
y_train_tr = torch.from_numpy(y_train).float()
y_test_tr = torch.from_numpy(y_test).float()

In [73]:
# After Converting its types
print("X_train : ",type(X_train_tr))
print("X_test : ",type(X_test_tr))

print("y_train : ",type(y_train_tr))
print("y_test : ",type(y_test_tr))


X_train :  <class 'torch.Tensor'>
X_test :  <class 'torch.Tensor'>
y_train :  <class 'torch.Tensor'>
y_test :  <class 'torch.Tensor'>


#####  Model Training Process

In [74]:
# Important Parameters
learning_rate = 0.1
epochs = 25

In [75]:
# Defining the model
class SimpleNNModel(nn.Module):
  def __init__(self,num_features):
    super().__init__()

    self.linear = nn.Linear(in_features=num_features,out_features=1)
    self.sigmoid = nn.Sigmoid()

  def forward(self,features):
    # It calculates z = w*x + b
    z = self.linear(features)
    # It applies sigmoid activation on the z
    y_pred = self.sigmoid(z)

    return y_pred

In [76]:
# Defining a loss function i,e Binary Cross Entropy Loss Function
loss_function = nn.BCELoss()

##### Training Pipeline

In [81]:
# create model
model = SimpleNNModel(X_train_tr.shape[1])

# Creating the optimzer using optim module(i,e Stochastic Gradient Descent) for the updation of the weights/parameters
optimizer = torch.optim.SGD(model.parameters(),lr=learning_rate)

##### Training model for the num of epochs
for epoch in range(epochs):

  # Forward pass
  y_pred = model(X_train_tr)

  # Loss Calculation
  loss = loss_function(y_pred,y_train_tr.view(-1,1))


  # Clearing the gradients so that it should'nt accumulate
  optimizer.zero_grad()

  # Backward pass
  loss.backward()

  # Parameters Updation
  optimizer.step()


  print(f"Epoch : {epoch + 1} , Loss : {loss:.4f}")


Epoch : 1 , Loss : 0.8038
Epoch : 2 , Loss : 0.5718
Epoch : 3 , Loss : 0.4551
Epoch : 4 , Loss : 0.3872
Epoch : 5 , Loss : 0.3426
Epoch : 6 , Loss : 0.3109
Epoch : 7 , Loss : 0.2870
Epoch : 8 , Loss : 0.2682
Epoch : 9 , Loss : 0.2530
Epoch : 10 , Loss : 0.2405
Epoch : 11 , Loss : 0.2298
Epoch : 12 , Loss : 0.2207
Epoch : 13 , Loss : 0.2127
Epoch : 14 , Loss : 0.2057
Epoch : 15 , Loss : 0.1994
Epoch : 16 , Loss : 0.1938
Epoch : 17 , Loss : 0.1888
Epoch : 18 , Loss : 0.1842
Epoch : 19 , Loss : 0.1800
Epoch : 20 , Loss : 0.1761
Epoch : 21 , Loss : 0.1726
Epoch : 22 , Loss : 0.1693
Epoch : 23 , Loss : 0.1662
Epoch : 24 , Loss : 0.1634
Epoch : 25 , Loss : 0.1607


In [82]:
with torch.no_grad():
  y_pred = model(X_test_tr)

  # As we are getting the y_pred in the range of 0 to 1, but we want binary 0 or 1 so we are normalizing it
  y_pred = (y_pred > 0.5).float()
  accuracy = (y_pred == y_test_tr).float().mean()

print(f"Accuracy : {accuracy.item():.4f}")

Accuracy : 0.5369
